# LPCMCI simulation baseline

Run this notebook independently of `oasis.ipynb`. Prepare the shared project environment once with `uv sync --frozen --all-extras`; the committed notebook extras preserve the required kernel tools, and selecting all extras prevents one baseline sync from removing the other. Raw PAG tensors are primary; the saved lagged skeleton is a lossy support-only projection. After the 5,000-fit matched grid, the notebook automatically runs the c-GC/c-GC* H1-H4 analysis suite: representative recovery, locked rise/fall tests, cyclic-shift and reverse-time nulls, the falling comparator, W_IC contrasts, and noise/frame-rate sweeps.

<!-- reviewer-resume-contract -->
## Execution and resume contract

Each outer-run–condition–seed unit is checkpointed. It recreates the exact static c-GC/c-GC* network and fluorescence input and records an input digest. Re-run the identical cell after interruption; do not change the grid, representations, or output directory while resuming. The main progress file ends at 1,000/1,000 units and 5,000 rows in `graph_recovery_rows.csv`; `full_analysis/progress.json` then ends at 224/224 diagnostic units and fits.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'simulation_baselines.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')


PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
RUNNER_ENV['PYTHONUNBUFFERED'] = '1'
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))
sys.path.insert(0, SOURCE_ROOT)

from calcium_transient_rising_flank.checkpointing import format_progress

NOTEBOOK_TAG = 'simulations/lpcmci.ipynb'
RUN_LPCMCI = True
N_RUNS_OUTER = 10
N_SEEDS = 20
N_STEPS = 3000
REPRESENTATIONS = 'full,deconvolved,rise,fall,fall_residual'
N_NULL = 6
OUTPUT_DIR = PACKAGE_ROOT / 'outputs/revision_campaign/lpcmci_simulation'
FULL_ANALYSIS_DIR = OUTPUT_DIR / 'full_analysis'
command = [
    RUNNER_PYTHON, 'examples/simulation_baselines.py',
    '--components', 'lpcmci',
    '--representations', REPRESENTATIONS,
    '--n-runs-outer', str(N_RUNS_OUTER), '--n-seeds', str(N_SEEDS),
    '--n-steps', str(N_STEPS), '--output-dir', str(OUTPUT_DIR), '--resume',
]
analysis_command = [
    RUNNER_PYTHON, 'examples/simulation_baseline_diagnostics.py',
    '--baseline', 'lpcmci', '--baseline-dir', str(OUTPUT_DIR),
    '--output-dir', str(FULL_ANALYSIS_DIR), '--n-null', str(N_NULL), '--resume',
]
progress_path = OUTPUT_DIR / 'progress.json'
analysis_progress_path = FULL_ANALYSIS_DIR / 'progress.json'
total_units = N_RUNS_OUTER * 5 * N_SEEDS
total_fits = total_units * len(REPRESENTATIONS.split(','))
analysis_units = len(REPRESENTATIONS.split(',')) + (N_NULL + 3) + N_RUNS_OUTER * min(3, N_SEEDS) * 7


def notebook_log(status: str, message: str) -> None:
    print(f'[{NOTEBOOK_TAG}] {status}: {message}', flush=True)


def show_resume_state(*, label: str, fallback_total: int, path: Path) -> None:
    if path.exists():
        saved = json.loads(path.read_text())
        completed = int(saved.get('completed_unit_count') or 0)
        expected = int(saved.get('expected_unit_count') or fallback_total or 1)
        total = max(expected, 1)
        notebook_log('RESUMED', format_progress(min(completed, total), total, label=label))
        notebook_log(
            'RESUMED',
            f"status={saved.get('status')} active={saved.get('active_unit')} progress_file={path}",
        )
    else:
        notebook_log('START', format_progress(0, max(fallback_total, 1), label=label))
        notebook_log('START', f'no saved progress at {path}')


notebook_log('START', f'output={OUTPUT_DIR}')
notebook_log('START', f'planned work={total_units} checkpoint units / {total_fits} LPCMCI fits')
show_resume_state(label='LPCMCI checkpoint units', fallback_total=total_units, path=progress_path)
notebook_log('START', f'launching: {shlex.join(command)}')
if RUN_LPCMCI:
    subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
    show_resume_state(label='LPCMCI checkpoint units', fallback_total=total_units, path=progress_path)
    notebook_log('DONE', 'matched-grid runner finished successfully')
    notebook_log('START', f'full H1-H4 analysis planned work={analysis_units} diagnostic fits')
    show_resume_state(label='LPCMCI full-analysis units', fallback_total=analysis_units, path=analysis_progress_path)
    notebook_log('START', f'launching: {shlex.join(analysis_command)}')
    subprocess.run(analysis_command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
    show_resume_state(label='LPCMCI full-analysis units', fallback_total=analysis_units, path=analysis_progress_path)
    notebook_log('DONE', 'full H1-H4 analysis finished successfully')

summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
analysis_summary_path = FULL_ANALYSIS_DIR / 'summary.json'
if analysis_summary_path.exists():
    analysis_summary = json.loads(analysis_summary_path.read_text())
    print(json.dumps(analysis_summary, indent=2))
